# Visualize a single frame

In [1]:
import numpy as np
import open3d as o3d
import matplotlib.pyplot as plt

# Path to the LiDAR frame
bin_path = "dataset/sequences/00/velodyne/000000.bin"

# 1. Load Data
points = np.fromfile(bin_path, dtype=np.float32).reshape(-1, 4)

# 2. Separate Coordinate and Intensity
xyz = points[:, :3]  # spatial coordinates
intensity = points[:, 3]  # reflection intensity

# 3. Create Open3D Point Cloud Object
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(xyz)

# --- Visualization: Color by Intensity ---
# Raw LiDAR is just black/white. Let's map intensity to color for better contrast.
# Normalize intensity to 0-1 range for color mapping
max_intensity = np.max(intensity)
# Avoid division by zero if max is 0
if max_intensity > 0:
    intensity_norm = intensity / max_intensity
else:
    intensity_norm = intensity

colormap = plt.get_cmap("viridis")
colors = colormap(intensity_norm)[:, :3]
pcd.colors = o3d.utility.Vector3dVector(colors)

# 4. Visualize
print(f"Loaded {len(points)} points. Opening viewer...")
print("Press 'h' in the window to see help (options for point size, etc.)")

# A coordinate frame to see origin (Red=X, Green=Y, Blue=Z)
mesh_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(
    size=2.0, origin=[0, 0, 0]
)

o3d.visualization.draw_geometries(
    [pcd, mesh_frame], window_name="SemanticKITTI Frame 00"
)

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Loaded 124668 points. Opening viewer...
Press 'h' in the window to see help (options for point size, etc.)


# Noise and Outlier Removal

In [2]:
# --- 1. Low-Z Noise Removal ---
z_threshold = -2.0  # Standard for KITTI (Sensor is ~1.73m above ground)

# Create the mask
mask = xyz[:, 2] > z_threshold

# Apply mask to BOTH points and colors
xyz_filtered = xyz[mask]
# We need to reshape or re-access the colors array to filter it
colors_filtered = np.asarray(pcd.colors)[mask]

# Update the Open3D object
pcd.points = o3d.utility.Vector3dVector(xyz_filtered)
pcd.colors = o3d.utility.Vector3dVector(colors_filtered)

print(f"Low-Z Filter: Reduced from {len(xyz)} to {len(xyz_filtered)} points")

# --- 2. Statistical Outlier Removal ---
# This function returns a NEW point cloud with colors preserved automatically
pcd_denoised, ind = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)

# Extract points for printing info
xyz_denoised = np.asarray(pcd_denoised.points)
print(
    f"Statistical Filter: Reduced from {len(xyz_filtered)} to {len(xyz_denoised)} points"
)

# --- 3. Visualization ---
o3d.visualization.draw_geometries(
    [pcd_denoised, mesh_frame],
    window_name="Filtered & Denoised Point Cloud",
)

Low-Z Filter: Reduced from 124668 to 116455 points
Statistical Filter: Reduced from 116455 to 112674 points


# Downsampling

In [3]:
voxel_size = 0.05  # 5 cm

pcd_down = pcd_denoised.voxel_down_sample(voxel_size=voxel_size)

xyz_down = np.asarray(pcd_down.points)

print("Before downsampling:", len(pcd_denoised.points), "points")
print("After downsampling :", len(pcd_down.points), "points")

o3d.visualization.draw_geometries([pcd_denoised], window_name="Before Downsampling")
o3d.visualization.draw_geometries([pcd_down], window_name="After Downsampling")

Before downsampling: 112674 points
After downsampling : 81478 points


# Ground Removal with RANSAC

In [4]:
# RANSAC for ground removal

# RANSAC parameters
distance_threshold = 0.2  # Max distance to plane (m)
ransac_n = 3  # Number of points that are randomly sampled to estimate a plane
num_iterations = 1000  # RANSAC iterations

print("Running RANSAC plane segmentation...")

# Fit the ground plane
plane_model, inliers = pcd_down.segment_plane(
    distance_threshold=distance_threshold,
    ransac_n=ransac_n,
    num_iterations=num_iterations,
)

[a, b, c, d] = plane_model
print(f"Plane equation: {a:.3f}x + {b:.3f}y + {c:.3f}z + {d:.3f} = 0")

# The ground plane normal (a, b, c) should be roughly (0, 0, 1).
# If 'c' is small (e.g., < 0.5), the normal is pointing sideways -> It's a WALL.
if abs(c) < 0.5:
    print("WARNING: The detected plane looks like a WALL, not the ground!")
    print("Recommendation: Adjust RANSAC parameters or crop the ROI further.")

# Extract ground and non-ground points
ground_pcd = pcd_down.select_by_index(inliers)
objects_pcd = pcd_down.select_by_index(inliers, invert=True)

# Color for visualization
ground_pcd.paint_uniform_color([0.5, 0.5, 0.5])  # gray ground
objects_pcd.paint_uniform_color([1.0, 0.0, 0.0])  # red objects

# Visualize
o3d.visualization.draw_geometries(
    [ground_pcd, objects_pcd],
    window_name="RANSAC Ground Removal",
)

Running RANSAC plane segmentation...
Plane equation: -0.012x + 0.027y + 1.000z + 1.753 = 0


# Clustering with DBSCAN

In [5]:
# DBSCAN for clustering

# DBSCAN clustering parameters
eps = 0.5  # 50 cm
min_points = 10  # Minimum points to form a cluster

print("Running DBSCAN...")

labels = np.array(
    objects_pcd.cluster_dbscan(eps=eps, min_points=min_points, print_progress=True)
)

max_label = labels.max()
print(f"point cloud has {max_label + 1} clusters")
colors = plt.get_cmap("tab20")(labels / (max_label if max_label > 0 else 1))
colors[labels < 0] = 0  # Noise set to black
objects_pcd.colors = o3d.utility.Vector3dVector(colors[:, :3])

o3d.visualization.draw_geometries(
    [ground_pcd, objects_pcd],
    window_name="DBSCAN Clusters",
)

Running DBSCAN...
point cloud has 175 clusters


# Bounding Boxes

In [6]:
bbox_objects = []

# We skip label -1 because that is noise in DBSCAN
unique_labels = np.unique(labels)

print(f"Analyzing {len(unique_labels) - 1} clusters for valid objects...")
min_volume_threshold = 0.5  # Minimum volume in cubic meters (m^3)
max_volume_threshold = 100  # Maximum volume in cubic meters

for label in unique_labels:
    if label == -1:
        continue  # Skip noise

    # --- Extract Points for this Cluster ---
    cluster_indices = np.asarray(labels == label).nonzero()[0]
    cluster_pcd = objects_pcd.select_by_index(cluster_indices)

    # --- Create Bounding Box ---
    points_in_cluster = np.asarray(cluster_pcd.points)
    if len(points_in_cluster) < 15:
        continue  # Skip tiny clusters
    bbox = cluster_pcd.get_oriented_bounding_box()
    bbox.color = (0, 1, 0)  # Color it Green

    # --- Filter out bushes, walls, and tiny artifacts ---
    # 1. Check Volume
    volume = bbox.volume()
    if volume < min_volume_threshold or volume > max_volume_threshold:
        continue

    # 2. Check Dimensions
    extent = bbox.extent
    sorted_ext = np.sort(extent)  # [Small, Medium, Large]
    min_dim, med_dim, max_dim = sorted_ext[0], sorted_ext[1], sorted_ext[2]

    # Remove "Walls" (Too long)
    if max_dim > 8.0:
        continue

    # Remove "Bushes/Trash" (Too small/flat)
    if max_dim < 0.5 or med_dim < 0.2:
        continue

    # Remove "Floating" Objects (Leaves/Signs)
    center = bbox.get_center()
    if center[2] > 0.5:
        continue  # Too high off the ground

    # If it passes the checks, add it to our list
    bbox_objects.append(bbox)

print(f"Filtered down to {len(bbox_objects)} valid objects.")

# --- Visualization ---
o3d.visualization.draw_geometries(
    [ground_pcd, objects_pcd] + bbox_objects,
    window_name="Bounding Boxes",
)

Analyzing 175 clusters for valid objects...
Filtered down to 58 valid objects.


# Tracking

In [7]:
import numpy as np
import scipy.spatial
from scipy.optimize import linear_sum_assignment


class CentroidTracker:
    def __init__(self, max_distance=2.0, max_disappeared=5):
        self.next_object_id = 0
        self.objects = {}  # {ID: np.array([x,y,z])}
        self.disappeared = {}  # {ID: count}
        self.max_distance = max_distance
        self.max_disappeared = max_disappeared

    def register(self, centroid):
        self.objects[self.next_object_id] = centroid
        self.disappeared[self.next_object_id] = 0  # Initialize counter
        self.next_object_id += 1

    def deregister(self, object_id):
        del self.objects[object_id]
        del self.disappeared[object_id]

    def update(self, new_bboxes):
        # 1. Get centroids from new boxes
        if len(new_bboxes) == 0:
            # If no new detections, mark all existing objects as 'disappeared'
            for obj_id in list(self.disappeared.keys()):
                self.disappeared[obj_id] += 1
                if self.disappeared[obj_id] > self.max_disappeared:
                    self.deregister(obj_id)
            return self.objects

        input_centroids = np.array([b.get_center() for b in new_bboxes])

        # If no tracking objects exist, register all
        if len(self.objects) == 0:
            for i in range(len(input_centroids)):
                self.register(input_centroids[i])
            return self.objects

        # 2. Match Old to New
        object_ids = list(self.objects.keys())
        object_centroids = list(self.objects.values())

        # Distance Matrix (Rows = Old, Cols = New)
        D = scipy.spatial.distance.cdist(np.array(object_centroids), input_centroids)

        # Hungarian Algorithm
        rows, cols = linear_sum_assignment(D)

        # Sets to keep track of what we have used
        used_rows = set(rows)
        used_cols = set(cols)

        # 3. Update Matched Objects
        for row, col in zip(rows, cols):
            # Retrieve the id using the row index
            obj_id = object_ids[row]

            if D[row, col] < self.max_distance:
                self.objects[obj_id] = input_centroids[col]  # Update position
                self.disappeared[obj_id] = 0  # Reset counter
            else:
                self.disappeared[obj_id] += 1  # Match found but too far -> Lost

        # 4. Handle UNMATCHED Old Objects (The ones Hungarian alg skipped)
        all_rows = set(range(len(object_ids)))
        unused_rows = all_rows - used_rows

        for row in unused_rows:
            obj_id = object_ids[row]
            self.disappeared[obj_id] += 1

        # 5. Handle UNMATCHED New Detections (Register as new)
        all_cols = set(range(len(input_centroids)))
        unused_cols = all_cols - used_cols

        for col in unused_cols:
            self.register(input_centroids[col])

        # 6. Clean up dead tracks
        # Create a copy of keys to modify dict while iterating
        for obj_id in list(self.disappeared.keys()):
            if self.disappeared[obj_id] > self.max_disappeared:
                self.deregister(obj_id)

        return self.objects

# Main loop

In [ ]:
import glob
import time

# 1. Setup File Paths
bin_paths = sorted(glob.glob("dataset/sequences/00/velodyne/*.bin"))[:30]

# 2. Initialize the Visualizer
vis = o3d.visualization.Visualizer()
vis.create_window(window_name="LiDAR Tracking Stream", width=1280, height=720)

# Setup render options
opt = vis.get_render_option()
opt.background_color = np.asarray([0, 0, 0])
opt.point_size = 2.0

# 3. Initialize Tracker
tracker = CentroidTracker(max_distance=2.0, max_disappeared=5)

# 4. Main Loop
print(f"Starting playback for {len(bin_paths)} frames...")

for frame_idx, bin_path in enumerate(bin_paths):

    # --- A. Load Data ---
    points = np.fromfile(bin_path, dtype=np.float32).reshape(-1, 4)
    xyz = points[:, :3]

    # Create base PointCloud
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(xyz)

    # --- B. Pipeline Execution ---

    # Step 1: Filter & Downsample
    mask = xyz[:, 2] > z_threshold
    xyz_filtered = xyz[mask]
    pcd.points = o3d.utility.Vector3dVector(xyz_filtered)
    pcd_denoised, ind = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
    pcd_down = pcd_denoised.voxel_down_sample(voxel_size=voxel_size)

    # Step 2: Ground Segmentation
    plane_model, inliers = pcd_down.segment_plane(
        distance_threshold=distance_threshold,
        ransac_n=ransac_n,
        num_iterations=num_iterations,
    )

    [a, b, c, d] = plane_model
    if abs(c) < 0.5:
        print("WARNING: The detected plane looks like a WALL, not the ground!")
        print("Recommendation: Adjust RANSAC parameters or crop the ROI further.")
    ground_pcd = pcd_down.select_by_index(inliers)
    objects_pcd = pcd_down.select_by_index(inliers, invert=True)
    ground_pcd.paint_uniform_color([0.5, 0.5, 0.5])  # gray ground
    objects_pcd.paint_uniform_color([1.0, 0.0, 0.0])  # red objects

    # Step 3: Clustering
    labels = np.array(
        objects_pcd.cluster_dbscan(eps=eps, min_points=min_points, print_progress=True)
    )
    max_label = labels.max()
    colors = plt.get_cmap("tab20")(labels / (max_label if max_label > 0 else 1))
    colors[labels < 0] = 0  # Noise set to black
    objects_pcd.colors = o3d.utility.Vector3dVector(colors[:, :3])

    # Step 4: Bounding Boxes
    bbox_objects = []
    unique_labels = np.unique(labels)

    for label in unique_labels:
        if label == -1:
            continue

        cluster_indices = np.asarray(labels == label).nonzero()[0]
        cluster_pcd = objects_pcd.select_by_index(cluster_indices)

        points_in_cluster = np.asarray(cluster_pcd.points)
        if len(points_in_cluster) < 15:
            continue
        bbox = cluster_pcd.get_oriented_bounding_box()
        bbox.color = (0, 1, 0)

        volume = bbox.volume()
        if volume < min_volume_threshold or volume > max_volume_threshold:
            continue
        extent = bbox.extent
        sorted_ext = np.sort(extent)
        min_dim, med_dim, max_dim = sorted_ext[0], sorted_ext[1], sorted_ext[2]
        if max_dim > 8.0:
            continue
        if max_dim < 0.5 or med_dim < 0.2:
            continue
        center = bbox.get_center()
        if center[2] > 0.5:
            continue
        # If it passes the checks, add it to our list
        bbox_objects.append(bbox)

    # Step 5: Tracking
    # Use the tracker to update IDs
    tracked_objects = tracker.update(bbox_objects)

    # --- C. Visualization Update ---
    # The visualizer holds "geometry" objects.
    # We need to clear the old ones and add the new ones for this frame.

    vis.clear_geometries()

    # Add the Points
    vis.add_geometry(ground_pcd)
    vis.add_geometry(objects_pcd)

    # Add the Bounding Boxes
    for bbox in bbox_objects:
        vis.add_geometry(bbox)

    # (Optional) Control the camera
    # ctr = vis.get_view_control()
    # ctr.rotate(5.0, 0.0) # Rotate camera slightly every frame for effect

    # Render this frame
    vis.poll_events()
    vis.update_renderer()

    # Print status to console
    print(f"Frame {frame_idx}: Tracking {len(tracker.objects)} objects")

    # Optional: Slow down if it's too fast
    # time.sleep(0.1)

# Keep window open after loop finishes
vis.run()

Starting playback for 30 frames...
Frame 0: Tracking 63 objects
Frame 1: Tracking 63 objects
Frame 2: Tracking 63 objects
Frame 3: Tracking 63 objects
Frame 4: Tracking 63 objects
Frame 5: Tracking 63 objects
Frame 6: Tracking 50 objects
Frame 7: Tracking 52 objects
Frame 8: Tracking 54 objects
Frame 9: Tracking 51 objects
Frame 10: Tracking 47 objects
Frame 11: Tracking 45 objects
Frame 12: Tracking 53 objects
Frame 13: Tracking 51 objects
Frame 14: Tracking 43 objects
Frame 15: Tracking 46 objects
Frame 16: Tracking 49 objects
Frame 17: Tracking 49 objects
Frame 18: Tracking 47 objects
Frame 19: Tracking 50 objects
Frame 20: Tracking 47 objects
Frame 21: Tracking 47 objects
Frame 22: Tracking 47 objects
Frame 23: Tracking 46 objects
Frame 24: Tracking 46 objects
Frame 25: Tracking 43 objects
Frame 26: Tracking 52 objects
Frame 27: Tracking 47 objects
Frame 28: Tracking 39 objects
Frame 29: Tracking 46 objects


: 